In [ ]:
import numpy as np

from typing import Union, List
from pymatching import Matching
import operators as o
import stabilizer_sim.decoding.nine_bit_rep_decoding as nr
import stabilizer_sim.decoding.three_bit_rep_decoding as tr
import decoding.utils as d

In [ ]:
rels = {
    ('X', 'Z'): '-Y',
    ('Z', 'X'): 'Y',
    ('X', 'Y'): 'Z',
    ('Y', 'X'): '-Z',
    ('Y', 'Z'): 'X',
    ('Z', 'Y'): '-X',
}

# TODO: Check assumptions
def get_syndrome_bit(state, stabilizer):
    bit = 0
    for comp in zip(state, stabilizer):
        if 'I' not in comp:
            if comp[0] != comp[1]:
                bit += 1

    return bit % 2

In [ ]:
rounds = 20
shots = 2000
px = 0.04
pm = 0.01

intial_state = o.Operator(3)

S1 = o.Z(3, [0, 1])
S2 = o.Z(3, [1, 2])

# stabs = [0, 0]
matching_graph = tr.build_3_qubit_matching_graph(rounds, 2, px, pm)
# er = {}


result = 0
for shot in range(shots):
    syndrome = []
    true_phys_error = [0, 0, 0]
    state = intial_state.copy()
    for rnd in range(rounds):

        
        for i in range(3):
            if np.random.random() < px:
                state = state * o.X(3, i)
                true_phys_error[i] += 1

        sb1 = get_syndrome_bit(state, S1)
        sb2 = get_syndrome_bit(state, S2)

        syndrome.append([sb1, sb2])

    
    detections = d.detect_changes(np.array(syndrome))
    # print(detections)
    matches = d.match_errors(matching_graph, detections)
    predicted_errors = d.extract_predicted_errors(matches)

    true_total_error = [i % 2 for i in true_phys_error]
    
    diff = sum(np.abs(true_total_error - predicted_errors))
    
    if diff != 0:
        result += 1

print(result/shots)
        

TypeError: extract_predicted_errors() missing 2 required positional arguments: 'num_stabilizers' and 'num_data_qubits'

In [ ]:
rounds = 20
shots = 10000
px = 0.04
pm = 0.00

intial_state = o.Operator(9)

stabs = {
    'S1': o.Z(9, [0, 1]),
    'S2': o.Z(9, [1, 2]),
    'S3': o.Z(9, [3, 4]),
    'S4': o.Z(9, [4, 5]),
    'S5': o.Z(9, [6, 7]),
    'S6': o.Z(9, [7, 8]),
    'S7': o.X(9, [1, 2, 3, 4, 5, 6]),
    'S8': o.X(9, [3, 4, 5, 6, 7, 8]),
    }

num_stabilizers = 8

# TODO: Need to treat as 4 separate graphs 
matching_graph = nr.build_9_qubit_matching_graph(rounds, px, pm)
# print(matching_graph)
# er = {}


result = 0
for shot in range(shots):
    syndrome = []
    true_phys_error = np.zeros(9)
    state = intial_state.copy()
    for rnd in range(rounds):
        # TODO: can do this more efficiently        
        for i in range(9):
            if np.random.random() < px:
                # print('hit', i)
                state = state * o.X(9, i)
                true_phys_error[i] += 1

        t_measurements = []
        for stab in stabs.keys():
            t_measurements.append(get_syndrome_bit(state, stabs[stab]))
        
        syndrome.append(t_measurements)

    detections = d.detect_changes(np.array(syndrome))
    # print(detections)
    matches = d.match_errors(matching_graph, detections)
    # print(matches)
    predicted_errors = d.extract_9_qubit_predicted_errors(matches)
    # print(predicted_errors)

    true_total_error = [i % 2 for i in true_phys_error]
    
    diff = sum(np.abs(true_total_error - predicted_errors))
    
    if diff != 0:
        result += 1

print(result/shots)

0.2433


In [ ]:
#Surface code
3*3

data_qubits = o.Operator(2)

# Maybe dont even need an initial state? Maybe don't need to measure stabilizers
# Can just directly add noise and generate what that would look like in the detectors/syndrome
# Generate blank syndrome
# [[0, 0, 0,]] 
# [[0, 0, 0,]]
# [[0, 0, 0,]]

# [[1, 1, 0,]] z11
# [[0, 0, 1,]] z10
# [[0, 0, 1,]]

TypeError: Operator.__init__() takes 2 positional arguments but 5 were given